<a href="https://colab.research.google.com/github/Samriddhi28-17/Text-to-Audio-withPython/blob/Samriddhi28-17/Text_to_audio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install PyMuPDF gTTS streamlit
!npm install localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 56.8 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.2.1
    Uninstalling click-8.2.1:
      Successfully uninstalled click-8.2.1
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
added 22 packages in 3s
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼

In [2]:
%%writefile app.py

Writing app.py


In [3]:
import fitz
from gtts import gTTS
import os
import streamlit as st
import time

In [4]:
# Function to extract text from PDF

def extract_text_from_pdf(pdf_path, pages=None):
    text = ""
    try:
        doc = fitz.open(pdf_path)
        if pages is None:
            pages_to_read = range(len(doc))
        else:
            pages_to_read = [p - 1 for p in pages if 0 < p <= len(doc)]
        for page_num in pages_to_read:
            page = doc.load_page(page_num)
            text += page.get_text()
        return text
    except Exception as e:
        return f"An error occurred: {e}"

In [5]:
# Function to convert text to audio

def convert_text_to_audio(text, output_file, lang='en'):
    try:
        tts = gTTS(text=text, lang=lang, slow=False)
        tts.save(output_file)
        print(f"Audio saved to {output_file}")
    except Exception as e:
        print(f"An error occurred during audio conversion: {e}")

In [6]:
st.title("PDF to Audiobook Converter 📚🎧")
st.markdown("Upload a PDF and convert its text to an audiobook. You can select specific pages to convert.")

2025-09-16 05:42:27.930 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-16 05:42:28.447 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2025-09-16 05:42:28.448 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-16 05:42:28.449 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-16 05:42:28.450 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-16 05:42:28.452 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-16 05:42:28.453 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()

In [7]:
# File uploader
uploaded_file = st.file_uploader("Choose a PDF file", type="pdf")

2025-09-16 05:44:00.884 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-16 05:44:00.886 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-16 05:44:00.887 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-16 05:44:00.888 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-16 05:44:00.889 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-16 05:44:00.890 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [8]:
if uploaded_file is not None:
    # Save the uploaded file to the disk
    with open("uploaded_pdf.pdf", "wb") as f:
        f.write(uploaded_file.getbuffer())

    # User options
    st.subheader("Conversion Options")
    conversion_option = st.radio("Choose conversion type:", ("Full book", "Custom pages"))

    pages_to_read = None
    if conversion_option == "Custom pages":
        pages_input = st.text_input("Enter pages to convert (e.g., 1, 3-5, 8)", "1")
        try:
            pages_to_read = []
            for part in pages_input.replace(" ", "").split(','):
                if '-' in part:
                    start, end = map(int, part.split('-'))
                    pages_to_read.extend(range(start, end + 1))
                else:
                    pages_to_read.append(int(part))
        except ValueError:
            st.error("Invalid page input. Please use a format like '1, 3-5, 8'.")
            st.stop()

    output_filename = "audiobook.mp3"

    if st.button("Generate Audiobook"):
        with st.spinner("Extracting text and converting to audio..."):
            extracted_text = extract_text_from_pdf("uploaded_pdf.pdf", pages=pages_to_read)

            if "An error occurred" in extracted_text:
                st.error(extracted_text)
            elif not extracted_text.strip():
                st.warning("No text was extracted from the specified pages. The PDF might be an image-based scan.")
            else:
                success = convert_text_to_audio(extracted_text, output_filename)

                if success:
                    st.success("Audiobook generated successfully! 🎉")
                    st.audio(output_filename)

                    with open(output_filename, "rb") as file:
                        st.download_button(
                            label="Download Audiobook",
                            data=file,
                            file_name=output_filename,
                            mime="audio/mpeg"
                        )

In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

⠙⠹

your url is: https://brown-goats-appear.loca.lt

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.86.236.45:8501

